# 简单装配线平衡问题 (SALBP)

**类别：** 调度

来源： [https://www.hexaly.com/templates/simple-assembly-line-balancing-problem-salbp](https://www.hexaly.com/templates/simple-assembly-line-balancing-problem-salbp)


## 问题

**在 Simple Assembly Line Balancing Problem 中**，我们考虑一组必须被分配到工作站中的任务。每个任务需要一定的处理时间。每个工作站中所有任务处理时间之和不能超过某个限制，称为节拍时间（cycle time）。任务之间还存在优先级约束。每个任务必须与其所有前驱任务位于同一工作台或更靠后的工作台。最后，目标函数是最小化工作站的数量。

	

### 学到的建模原则

- 添加 [set decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模工作站的内容
- 定义 [lambda function](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算每个工作站的总处理时间
- 借助 ‘[find](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#find)‘ 算子检索每个任务所在工作站的索引


## 数据

我们提供的 Simple Assembly Line Balancing Problem (SALBP) 实例来自 [Otto et al.](https://assembly-line-balancing.de/salbp/benchmark-data-sets-2013/)，格式如下：

- 任务数量
- 节拍时间限制
- 对于每个任务，其索引和处理时间
- 对于每个优先级约束，前驱任务的索引和后继任务的索引


## 模型

Simple Assembly Line Balancing Problem (SALBP) 的 Hexaly 模型使用 [set decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)。每个 set 表示分配到一个工作站的任务。借助 **partition** 算子，我们确保每个任务被分配到恰好一个工作站。

每个工作站的总处理时间通过 [lambda function](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 计算，对所有已分配任务应用 **sum** 算子。注意此求和中项的数量以及 set 的大小在搜索过程中是变化的。然后我们可以约束该数量不超过节拍时间。

使用 [**find**](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#find) 算子，我们可以检索为处理每个任务而选择的工作站的索引。这使我们能够写出任务之间的优先级约束：每个任务应位于其所有后继任务所在工作站之前的某个工作站。

我们计算已使用的工作站数量，它等于非空 set 变量的数量，并将其最小化。


## Results

在由 1,000 个任务实例组成的文献大型基准上，Hexaly Optimizer 在 1 分钟运行时间内对 Simple Assembly Line Balancing Problem (SALBP) 达到了 **0.3% 的平均差距**。我们的 [Simple Assembly Line Balancing Problem (SALBP) benchmark page](https://www.hexaly.com/benchmark/hexaly-vs-gurobi-vs-cpo-simple-assembly-line-balancing-problem-salbp) 展示了 Hexaly Optimizer 在这一具有挑战性的问题上如何超越 Gurobi 和 CP Optimizer 等传统通用优化求解器。

[Explore this benchmark](https://www.hexaly.com/benchmark/hexaly-vs-gurobi-vs-cpo-simple-assembly-line-balancing-problem-salbp)


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys


#
# Functions to read the instances
#
def read_elem(filename):
    with open(filename) as f:
        return [str(elem) for elem in f.read().split()]


def read_instance(instance_file):
    file_it = iter(read_elem(instance_file))

    for _ in range(3):
        next(file_it)

    # Read number of tasks
    nb_tasks = int(next(file_it))
    max_nb_stations = nb_tasks
    for _ in range(2):
        next(file_it)

    # Read the cycle time limit
    cycle_time = int(next(file_it))
    for _ in range(5):
        next(file_it)

    # Read the processing times
    processing_time_dict = {}
    for _ in range(nb_tasks):
        task = int(next(file_it)) - 1
        processing_time_dict[task] = int(next(file_it))
    for _ in range(2):
        next(file_it)
    processing_time = [elem[1] for elem in sorted(processing_time_dict.items(),
                                                  key=lambda x: x[0])]

    # Read the successors' relations
    successors = {}
    while True:
        try:
            pred, succ = next(file_it).split(',')
            pred = int(pred) - 1
            succ = int(succ) - 1
            if pred in successors:
                successors[pred].append(succ)
            else:
                successors[pred] = [succ]
        except:
            break
    return nb_tasks, max_nb_stations, cycle_time, processing_time, successors


def main(instance_file, output_file, time_limit):
    nb_tasks, max_nb_stations, cycle_time, processing_time_data, \
        successors_data = read_instance(instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Decision variables: station_vars[s] is the set of tasks assigned to station s
        station_vars = [model.set(nb_tasks) for s in range(max_nb_stations)]
        stations = model.array(station_vars)
        model.constraint(model.partition(stations))

        # Objective: nb_used_stations is the total number of used stations
        nb_used_stations = model.sum(
            (model.count(station_vars[s]) > 0) for s in range(max_nb_stations))

        # All stations must respect the cycleTime constraint
        processing_time = model.array(processing_time_data)
        time_lambda = model.lambda_function(lambda i: processing_time[i])
        time_in_station = [model.sum(station_vars[s], time_lambda)
                           for s in range(max_nb_stations)]
        for s in range(max_nb_stations):
            model.constraint(time_in_station[s] <= cycle_time)

        # The stations must respect the succession's order of the tasks
        task_station = [model.find(stations, i) for i in range(nb_tasks)]
        for i in range(nb_tasks):
            if i in successors_data.keys():
                for j in successors_data[i]:
                    model.constraint(task_station[i] <= task_station[j])

        # Minimization of the number of active stations
        model.minimize(nb_used_stations)

        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = time_limit

        optimizer.solve()

        # Write the solution in a file following the format:
        # - 1st line: value of the objective
        # - 2nd line: number of tasks
        # - following lines: task's number, station's number
        if output_file is not None:
            with open(output_file, 'w') as f:
                f.write("%d\n" % nb_used_stations.value)
                f.write("%d\n" % nb_tasks)
                for i in range(nb_tasks):
                    f.write("{},{}\n".format(i + 1, task_station[i].value + 1))


if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python assembly_line_balancing.py instance_file \
            [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 20
    main(instance_file, output_file, time_limit)
